# Cross-dataset metrics: brains versus phantoms

This notebook is one of 5 focused notebooks split out of the original `esmrmb_variant_comparison.ipynb` (archived under `_archive/`) so each analysis stage can be opened and run on its own. No analysis or plotting logic changed in the split -- every cell below is copied verbatim from the original notebook; only the output path gained a `resample-ogse_rest_rician` segment naming the resampling model used (`build_resampled_contrasts`, fit with `M_ogse_rest_rician`), so a future alternative resampling model can sit alongside this one without collisions.

The sibling notebooks are: `signal_and_contrast_panels.ipynb`, `contrast_vs_lcf_panels.ipynb`, `dataset_metrics.ipynb`, `contrast_models_overlap.ipynb`, `cross_dataset_metrics.ipynb`.

## 1. Imports, provenance, and data model

Run the notebook from any directory inside the project. The helper below finds the project root without relying on a user-specific absolute path. All scientific calculations are implemented in `src/presentation_analysis/abstract_figures.py`; the notebook supplies configuration, displays intermediate QC, and exports reproducible artifacts.

The long master table is the source of truth. A signal row is identified by acquisition metadata (`subj`, `sheet`, $T_D$, $N$, source file and gradient step), spatial metadata (`roi`, rotated `direction`), statistic (`avg` or `std`), and processing provenance (`variant`, `analysis_tag`, `row_kind`). `signal_rotated` means the diffusion tensor and signal directions have already been expressed relative to the local ROI/fiber frame. The analysis never infers missing rows and never merges subjects or acquisition sheets.

Main units used below: $T_D$, $\tau_c$, $\tau_f$, $c$, and $\delta$ are in ms; gradient strength is in mT/m; $D_0$ is stored in m$^2$/ms; and the Capiglioni filter length $l_{cf}=\sqrt{D_0\tau_f}$ is displayed in $\mu$m. Normalized signal and contrast are dimensionless.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'Data-BIDS').is_dir() and (candidate / 'repos' / 'signal_analysis').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the Project-Balseiro-Microstructure root.')


PROJECT_ROOT = find_project_root()
REPO_ROOT = PROJECT_ROOT / 'repos' / 'signal_analysis'
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Reload local helpers so Run All also picks up code changes in an existing kernel.
import presentation_analysis.abstract_figures as _abstract_figures
import presentation_analysis as _presentation_analysis
importlib.reload(_abstract_figures)
importlib.reload(_presentation_analysis)

from presentation_analysis import (
    CC_ROIS,
    aggregate_alpha,
    build_resampled_contrasts,
    compare_with_reference,
    compare_with_reference_metrics,
    discover_analysis_runs,
    export_all_contrast_lcf_panels,
    export_all_signal_contrast_panels,
    fit_tc_pseudohuber,
    load_alpha_summaries,
    load_masters,
    master_qc_summary,
    merge_alpha_delta,
    normalize_alpha_to_internal_reference,
    plot_alpha_delta_scatter,
    plot_contrast_lcf_grid,
    plot_metric_comparison,
    plot_signal_contrast_example,
    save_figure_formats,
    summarize_contrast_shape,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 80)
PROJECT_ROOT

## 2. Configuration

The first lines under **Quick selection** are normally the only ones that need editing. Choose `DATASET = 'brains'` or `DATASET = 'phantoms'`, then set `SUBJECTS_OVERRIDE`, `ROI_OVERRIDE`, and `DIRECTION_OVERRIDE`. Use a Python list even for one value, for example `['20220610P']`, `['fiber1']`, or `['long']`; use `None` to include every value allowed by the dataset profile. `SHEETS_OVERRIDE` can isolate an acquisition when one subject has several sheets.

For phantoms, selecting exactly one known subject also selects its processed source automatically: `20220610P` uses `den_gr--manual`, whereas `20260706P` uses `den_gr-topup--manual`. `DWI_LEVEL_OVERRIDE` is therefore normally left as `None`. `EXAMPLE_*` and `EXAMPLE_TD_MS` change only the detailed example panels; they do not filter the complete analysis. `VARIANTS = None` discovers current and future ROI variants automatically.

The reference diffusivity $D_0$ is a physical prior used by the restricted-signal model and by the gradient-to-length transformation; it is not refitted independently for every ROI in this notebook. Consequently, brain and phantom values must remain explicit and should be reviewed before the phantom analysis is finalized. The $N=8$ and $N=4$ selection defines the two encoding waveforms whose difference is studied. `GRADIENT_COLUMN = 'g'` is the project default because it preserves the recorded, non-equally-spaced b-value sampling; use `g_thorsten` only for an explicit legacy sensitivity analysis.

`SAVE_OUTPUTS` controls persistent tables and presentation figures. Every saved figure is written in both PNG and PDF. `EXPORT_ALL_SIGNAL_CONTRAST_PANELS` controls the exhaustive Section 6 gallery. `EXPORT_ALL_CONTRAST_LCF_PANELS` creates a Figure-02-style image for every selected subject and ROI, with directions in columns and every available $T_D$ as a colored curve. Both are enabled by default.

In [ ]:
# -----------------------------------------------------------------------------
# QUICK SELECTION: edit this block first, then use Kernel > Restart & Run All.
# -----------------------------------------------------------------------------
DATASET = 'brains'  # 'brains' or 'phantoms'
#Examples: ['20220610P'], ['20260706P'], ['ADBN'], ['ADBN', 'ARVE', 'BRAIN', 'LUDG', 'MBBL', 'SNVN'], or None for all subjects
SUBJECTS_OVERRIDE = None
# Optional acquisition-sheet filter, useful when one brain subject has several scans
SHEETS_OVERRIDE = None            # e.g. ['20230623_BRAIN-4']
# Lists filter the complete analysis; None uses all defaults/available values
ROI_OVERRIDE = None               # phantom e.g. ['fiber1']; brain e.g. ['PostCC']
DIRECTION_OVERRIDE = None         # e.g. ['long'], ['tra'], or ['long', 'tra']

# One detailed signal/contrast panel (these settings do NOT filter the analysis)
EXAMPLE_SUBJECT_OVERRIDE = None   # None uses the selected subject when it is unique
EXAMPLE_ROI_OVERRIDE = None       # e.g. 'fiber1' or 'PostCC' (a string, not a list)
EXAMPLE_DIRECTION_OVERRIDE = None # e.g. 'long' or 'tra'
EXAMPLE_TD_MS = 143.4             # old phantom also has 142.5 ms; nearest value is used

# Dataset profiles: acquisition physics and default selections. Usually do not edit.
DATASET_PROFILES = {
    'brains': {
        'dwi_level': 'den_gr-topup',
        'd0_m2_ms': 3.2e-12,     # 0.0032 mm2/s
        'rois': list(CC_ROIS),
        'directions': ['long', 'tra'],
        'example_subject': 'ADBN',
        'example_roi': 'PostCC',
        'example_direction': 'long',
    },
    'phantoms': {
        'dwi_level': 'den_gr-topup',
        'd0_m2_ms': 2.3e-12,     # 0.0023 mm2/s
        'rois': None,               # None uses every ROI/sample found in the phantom master
        'directions': None,         # None prefers long/tra, then uses every available direction
        'example_subject': '20220610P',
        'example_roi': None,
        'example_direction': None,
    },
}

# Advanced data-source selection
SEQUENCE = 'ogse'
# Leave None: the notebook maps 20220610P -> den_gr and 20260706P -> den_gr-topup.
DWI_LEVEL_OVERRIDE = None          # force only when auditing a specific processing level
VARIANTS = None                   # brains e.g. ['plain'] or ['plain', 'erode1', 'sket1']
REFERENCE_VARIANT = None           # None prefers plain; phantoms use manual

# Contrast definition from the abstract
N_HIGH = 8
N_LOW = 4
# Use the gradient derived from each recorded b-value. This preserves non-equally-spaced
# acquisitions such as ADBN, ARVE, SNVN, and 20260706P. Set 'g_thorsten' only to
# reproduce the legacy equally-spaced-axis analysis.
GRADIENT_COLUMN = 'g'
APPLY_GRADIENT_CORRECTION = True
RESAMPLE_GRID_SIZE = 1000
SIGNAL_TC_BOUNDS_MS = (0.1, 10000.0)
RICIAN_C_BOUNDS = (0.0, 2.0)
SIGNAL_TC_MODE = 'separate'         # Flexible curve construction; not a physical two-tc interpretation

# Example-panel display limits
LCF_LIMITS_UM = (2.5, 11.0)  # Historical Capiglioni/abstract axis
TD_COLOR_TOLERANCE_MS = 8.0  # Includes both historical and current acquisition times

# Pseudo-Huber configuration used in the abstract analysis
EXCLUDE_TD_MS = (76.0,)
TD_RANGE_MS = (50.0, 250.0)
C_BOUNDS_MS = (0.0, 10.0)
DELTA_BOUNDS_MS = (1e-6, 10000.0)

# Transparent display-only QC. All fits remain in the exported audit tables.
MIN_PSEUDOHUBER_R2_FOR_SUMMARY = 0.50
EXCLUDE_DELTA_BOUNDARY_FROM_SUMMARY = True

# Exhaustive Section 6 export: one panel per subject/sheet/ROI/direction/TD combination.
EXPORT_ALL_SIGNAL_CONTRAST_PANELS = True
# Figure-02 gallery: one contrast-vs-filtered-length image per subject and ROI.
EXPORT_ALL_CONTRAST_LCF_PANELS = True
CLEAN_BATCH_PANEL_DIR = True       # Remove stale PNG/PDF files and the manifest inside each gallery
BATCH_PANEL_DPI = 180             # Lower than the presentation figures to control disk usage
BATCH_PROGRESS_EVERY = 25

SAVE_OUTPUTS = True

# Known phantom subjects use different processed derivatives.
PHANTOM_DWI_LEVEL_BY_SUBJECT = {
    '20220610P': 'den_gr',
    '20260706P': 'den_gr-topup',
}

def selection_list(value):
    if value is None:
        return None
    return [value] if isinstance(value, str) else list(value)

SELECTED_SUBJECTS = selection_list(SUBJECTS_OVERRIDE)
SELECTED_SHEETS = selection_list(SHEETS_OVERRIDE)
if DATASET not in DATASET_PROFILES:
    raise ValueError(f'Unknown DATASET={DATASET!r}; choose one of {list(DATASET_PROFILES)}.')
PROFILE = DATASET_PROFILES[DATASET]
if DWI_LEVEL_OVERRIDE is not None:
    DWI_LEVEL = DWI_LEVEL_OVERRIDE
elif DATASET == 'phantoms' and SELECTED_SUBJECTS is not None and len(SELECTED_SUBJECTS) == 1:
    DWI_LEVEL = PHANTOM_DWI_LEVEL_BY_SUBJECT.get(SELECTED_SUBJECTS[0], PROFILE['dwi_level'])
else:
    DWI_LEVEL = PROFILE['dwi_level']
D0_REFERENCE_M2_MS = PROFILE['d0_m2_ms']
OUTPUT_DIR = (
    REPO_ROOT / 'notebooks' / 'outputs' / 'cross_dataset_metrics' / 'resample-ogse_rest_rician'
    / DATASET / DWI_LEVEL
)
OUTPUT_DIR

## 11. Phantom-to-phantom and brain-to-phantom comparison

This section is independent of the single-dataset/variant workflow above. It loads both available phantom acquisitions, compares `20220610P` with `20260706P`, and then compares a selectable phantom (default `20260706P`) with a selectable brain ROI variant (default `plain`). The phantom analysis uses the available `den_gr` and `den_gr-topup` products rather than pretending that the two acquisitions are segmentation variants.

Legacy phantom alpha workbooks may have been generated with the brain reference diffusivity (0.0032 mm$^2$/s). The pipeline now uses 0.0023 mm$^2$/s for phantoms, and this notebook independently audits and harmonizes loaded alpha values from their stored `D0_mean_mm2_s`. When direct rotated and legacy reconstructed `long`/`tra` rows coexist, the direct rotated row is retained. Any harmonization is recorded in the exported tables.

The primary alternatives to the peak-only metric use the entire positive contrast curve: normalized-gradient area, centroid and spread, plus a log-length centroid and geometric width. These are descriptive observables, not additional independent samples. The historical peak and pseudo-Huber results remain available as sensitivity analyses.

In [ ]:
# Cross-dataset selection. This is separate from the main selection in Section 2.
# Paths for known phantoms; edit only when a new processed phantom is added.
PHANTOM_RUN_CONFIG = {
    '20220610P': {'dwi_level': 'den_gr', 'variant': 'manual'},
    '20260706P': {'dwi_level': 'den_gr-topup', 'variant': 'manual'},
}
# Include both IDs for the phantom-to-phantom plots, or one ID to inspect one phantom.
PHANTOM_IDS = ['20220610P', '20260706P']
# Brain source/segmentation. Try 'erode1' or 'sket1' for sensitivity analyses.
BRAIN_DWI_LEVEL = 'den_gr-topup'
BRAIN_VARIANT = 'plain'
# None uses all brains; a list restricts the complete cross-dataset analysis.
BRAIN_SUBJECTS = None              # e.g. ['ADBN'] or ['ADBN', 'ARVE']
# Choose which loaded phantom is placed beside brains in Sections 11.3-11.4.
BRAIN_COMPARISON_PHANTOM = '20220610P'
PHANTOM_D0_M2_MS = 2.3e-12  # 0.0023 mm2/s
BRAIN_D0_M2_MS = 3.2e-12    # 0.0032 mm2/s
# Phantom ROIs used in the cross-cohort figures; water remains an alpha reference.
CROSS_COMPARISON_ROIS = ['fiber1', 'fiber2']
RUN_SHARED_TC_SENSITIVITY = True   # False makes Section 11 faster but skips model audit
CROSS_OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'cross_dataset_metrics' / 'resample-ogse_rest_rician'

if BRAIN_COMPARISON_PHANTOM not in PHANTOM_IDS:
    raise ValueError('BRAIN_COMPARISON_PHANTOM must be one of PHANTOM_IDS.')


In [ ]:
def run_domain_analysis(
    dataset, dwi_level, variant, d0_m2_ms, rois, internal_reference_rois, subjects=None
):
    domain_runs = discover_analysis_runs(
        PROJECT_ROOT, dataset=dataset, sequence='ogse', dwi_level=dwi_level
    )
    domain_master = load_masters(domain_runs, [variant])
    domain_alpha_raw = load_alpha_summaries(
        domain_runs, [variant], reference_d0_m2_ms=d0_m2_ms
    )
    if subjects is not None:
        domain_master = domain_master[domain_master['subj'].isin(subjects)].copy()
        domain_alpha_raw = domain_alpha_raw[domain_alpha_raw['subj'].isin(subjects)].copy()
    domain_alpha_internal = normalize_alpha_to_internal_reference(
        domain_alpha_raw, reference_rois=internal_reference_rois
    )
    internal_keys = ['variant', 'subj', 'roi', 'direction']
    domain_alpha_internal = domain_alpha_internal.groupby(internal_keys, as_index=False).agg(
        alpha_internal=('alpha_internal', 'mean'),
        alpha_internal_error=('alpha_internal_error', lambda x: (np.sqrt(np.nansum(x**2)) / x.notna().sum()) if x.notna().any() else np.nan),
        internal_reference_D0_mm2_s=('internal_reference_D0_mm2_s', 'mean'),
    )
    domain_alpha = aggregate_alpha(domain_alpha_raw)
    separate_contrast, separate_fits = build_resampled_contrasts(
        domain_master, d0_m2_ms=d0_m2_ms, n_high=N_HIGH, n_low=N_LOW,
        directions=['long', 'tra'], rois=rois,
        gradient_column=GRADIENT_COLUMN,
        apply_gradient_correction=APPLY_GRADIENT_CORRECTION,
        tc_bounds_ms=SIGNAL_TC_BOUNDS_MS, rician_c_bounds=RICIAN_C_BOUNDS,
        grid_size=RESAMPLE_GRID_SIZE, tc_mode='separate',
    )
    shape = summarize_contrast_shape(separate_contrast)
    shared_fits = pd.DataFrame()
    if RUN_SHARED_TC_SENSITIVITY:
        _, shared_fits = build_resampled_contrasts(
            domain_master, d0_m2_ms=d0_m2_ms, n_high=N_HIGH, n_low=N_LOW,
            directions=['long', 'tra'], rois=rois,
            gradient_column=GRADIENT_COLUMN,
            apply_gradient_correction=APPLY_GRADIENT_CORRECTION,
            tc_bounds_ms=SIGNAL_TC_BOUNDS_MS, rician_c_bounds=RICIAN_C_BOUNDS,
            grid_size=128, tc_mode='shared',
        )
    return {
        'runs': domain_runs, 'master': domain_master, 'alpha': domain_alpha,
        'alpha_internal': domain_alpha_internal,
        'contrast': separate_contrast, 'fits': separate_fits,
        'shape': shape, 'shared_fits': shared_fits,
    }

phantom_results = {}
for phantom_id in PHANTOM_IDS:
    config = PHANTOM_RUN_CONFIG[phantom_id]
    result = run_domain_analysis(
        'phantoms', config['dwi_level'], config['variant'],
        PHANTOM_D0_M2_MS, CROSS_COMPARISON_ROIS,
        ['water', 'water1', 'water2', 'water3'], [phantom_id],
    )
    for name in ['master', 'alpha', 'alpha_internal', 'contrast', 'fits', 'shape', 'shared_fits']:
        if not result[name].empty:
            result[name].insert(0, 'cohort', phantom_id)
    phantom_results[phantom_id] = result

brain_result = run_domain_analysis(
    'brains', BRAIN_DWI_LEVEL, BRAIN_VARIANT, BRAIN_D0_M2_MS,
    list(CC_ROIS), ['Left-Lateral-Ventricle', 'Right-Lateral-Ventricle'], BRAIN_SUBJECTS,
)
for name in ['master', 'alpha', 'alpha_internal', 'contrast', 'fits', 'shape', 'shared_fits']:
    if not brain_result[name].empty:
        brain_result[name].insert(0, 'cohort', 'brains')

phantom_shape = pd.concat([result['shape'] for result in phantom_results.values()], ignore_index=True)
phantom_alpha = pd.concat([result['alpha'] for result in phantom_results.values()], ignore_index=True)
phantom_alpha_internal = pd.concat(
    [result['alpha_internal'] for result in phantom_results.values()], ignore_index=True
)
display(phantom_shape.groupby(['cohort', 'roi', 'direction']).size().rename('curves').reset_index())


### 11.1 Model adequacy and peak identifiability

A shared effective $\tau_c$ is the physically constrained version of the single-restriction model; the separate-$\tau_c$ fit is retained only as a flexible interpolator for constructing a smooth contrast. BIC compares these nested descriptions per signal pair. Strong preference for separate $\tau_c$ indicates model mismatch or heterogeneous dynamics, not evidence that the tissue has two encoding-specific correlation times. A peak at the supported-gradient boundary is not an identified interior maximum and must not be converted into a microstructural time.

In [ ]:
def compare_tc_parameterizations(result):
    if result['shared_fits'].empty:
        return pd.DataFrame()
    keys = ['cohort', 'subj', 'sheet', 'roi', 'direction', 'td_ms']
    shared = result['shared_fits'][keys + ['ok', 'r2', 'rmse', 'bic', 'tc_shared_ms', 'peak_at_boundary']]
    separate = result['fits'][keys + ['ok', 'r2', 'rmse', 'bic', 'tc_high_ms', 'tc_low_ms', 'peak_at_boundary']]
    table = shared.merge(separate, on=keys, suffixes=('_shared', '_separate'), validate='one_to_one')
    table['delta_bic_shared_minus_separate'] = table['bic_shared'] - table['bic_separate']
    table['bic_preference'] = np.select(
        [table['delta_bic_shared_minus_separate'] < -6, table['delta_bic_shared_minus_separate'] > 6],
        ['shared', 'separate'], default='inconclusive',
    )
    return table

all_domain_masters = pd.concat(
    [*[result['master'] for result in phantom_results.values()], brain_result['master']],
    ignore_index=True,
)
protocol_rows = all_domain_masters[
    all_domain_masters['row_kind'].eq('signal_rotated')
    & all_domain_masters['stat'].eq('avg')
    & pd.to_numeric(all_domain_masters['N'], errors='coerce').isin([N_HIGH, N_LOW])
].copy()
protocol_audit = protocol_rows.groupby(
    ['cohort', 'subj', 'sheet', 'td_ms', 'N'], as_index=False
).agg(TE_ms=('TE', 'median'), TR_ms=('TR', 'median'), g_max_mTm=('g_max', 'max'))
protocol_audit = protocol_audit.pivot(
    index=['cohort', 'subj', 'sheet', 'td_ms'], columns='N',
    values=['TE_ms', 'TR_ms', 'g_max_mTm'],
).reset_index()
protocol_audit.columns = [
    '_'.join(str(part) for part in column if str(part) != '').rstrip('_')
    if isinstance(column, tuple) else str(column) for column in protocol_audit.columns
]
protocol_audit['TE_difference_high_minus_low_ms'] = (
    protocol_audit[f'TE_ms_{N_HIGH}'] - protocol_audit[f'TE_ms_{N_LOW}']
)
display(protocol_audit)

model_audit = pd.concat(
    [compare_tc_parameterizations(result) for result in [*phantom_results.values(), brain_result]],
    ignore_index=True,
)
model_audit_summary = model_audit.groupby('cohort', as_index=False).agg(
    signal_pairs=('subj', 'size'),
    shared_median_r2=('r2_shared', 'median'),
    separate_median_r2=('r2_separate', 'median'),
    strong_separate_fraction=('bic_preference', lambda x: (x == 'separate').mean()),
    separate_peak_boundary_fraction=('peak_at_boundary_separate', 'mean'),
)
display(model_audit_summary)


In [ ]:
shape_metrics_to_plot = [
    ('lcf_centroid_um', 'Contrast-weighted $l_{cf}$ centroid [$\mu$m]'),
    ('contrast_peak', 'Peak contrast'),
    ('contrast_auc_normalized_g', 'Contrast area on normalized gradient'),
]
fig, axes = plt.subplots(3, 2, figsize=(13, 12), sharex=False)
colors = {'20220610P': '#0072B2', '20260706P': '#D55E00'}
linestyles = {'fiber1': '-', 'fiber2': '--'}
for row, (metric, ylabel) in enumerate(shape_metrics_to_plot):
    for col, direction in enumerate(['long', 'tra']):
        ax = axes[row, col]
        subset = phantom_shape[phantom_shape['direction'].eq(direction)]
        for (cohort, roi), group in subset.groupby(['cohort', 'roi']):
            group = group.sort_values('td_ms')
            ax.plot(group['td_ms'], group[metric], marker='o', color=colors[cohort],
                    linestyle=linestyles[roi], label=f'{cohort} {roi}')
        ax.set_title(direction)
        ax.set_ylabel(ylabel)
        ax.set_xlabel('$T_D$ [ms]')
        ax.grid(alpha=0.25)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.suptitle('Phantom acquisition comparison: full-curve observables', y=0.985)
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.955),
           ncol=4, frameon=False)
fig.tight_layout(rect=(0.02, 0.02, 0.98, 0.90), h_pad=2.0, w_pad=2.0)
if SAVE_OUTPUTS:
    CROSS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_figure_formats(fig, CROSS_OUTPUT_DIR / '11_phantom_comparison_shape_metrics.png', dpi=300)
plt.show()


In [ ]:
selected_phantom_shape = phantom_results[BRAIN_COMPARISON_PHANTOM]['shape'].copy()
brain_shape = brain_result['shape'].copy()
comparison_shape = pd.concat([brain_shape, selected_phantom_shape], ignore_index=True)
comparison_alpha = pd.concat([brain_result['alpha'], phantom_results[BRAIN_COMPARISON_PHANTOM]['alpha']], ignore_index=True)
comparison_alpha_internal = pd.concat(
    [brain_result['alpha_internal'], phantom_results[BRAIN_COMPARISON_PHANTOM]['alpha_internal']],
    ignore_index=True,
)

fig, axes = plt.subplots(3, 2, figsize=(14, 12.5))
for row, (metric, ylabel) in enumerate(shape_metrics_to_plot):
    for col, direction in enumerate(['long', 'tra']):
        ax = axes[row, col]
        brain = brain_shape[brain_shape['direction'].eq(direction)]
        brain_summary = brain.groupby(['roi', 'td_ms'], as_index=False)[metric].agg(['mean', 'sem']).reset_index()
        for roi, group in brain_summary.groupby('roi'):
            ax.plot(group['td_ms'], group['mean'], color='0.65', alpha=0.8, linewidth=1, label=f'brain {roi}')
        phantom = selected_phantom_shape[selected_phantom_shape['direction'].eq(direction)]
        for roi, group in phantom.groupby('roi'):
            group = group.sort_values('td_ms')
            ax.plot(group['td_ms'], group[metric], marker='o', linewidth=2.5,
                    label=f'{BRAIN_COMPARISON_PHANTOM} {roi}')
        ax.set_title(direction)
        ax.set_ylabel(ylabel)
        ax.set_xlabel('$T_D$ [ms]')
        ax.grid(alpha=0.25)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.suptitle(f'Brains ({BRAIN_VARIANT}) versus {BRAIN_COMPARISON_PHANTOM}', y=0.985)
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.95),
           ncol=4, frameon=False)
fig.tight_layout(rect=(0.02, 0.02, 0.98, 0.93), h_pad=2.0, w_pad=2.0)
if SAVE_OUTPUTS:
    save_figure_formats(fig, CROSS_OUTPUT_DIR / '12_brain_vs_selected_phantom_shape_metrics.png', dpi=300)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(15, 10.5), sharey='row')
alpha_panels = [
    (comparison_alpha, 'alpha_macro', r'$\alpha_{external}=D_{eff}/D_{0,configured}$'),
    (comparison_alpha_internal, 'alpha_internal', r'$\alpha_{internal}=D_{eff}/D_{water,same scan}$'),
]
for row, (alpha_table, alpha_column, ylabel) in enumerate(alpha_panels):
  alpha_display = alpha_table[alpha_table['direction'].isin(['long', 'tra'])].copy()
  alpha_display['group'] = alpha_display['cohort'] + ' | ' + alpha_display['roi']
  for ax, direction in zip(axes[row], ['long', 'tra']):
    subset = alpha_display[alpha_display['direction'].eq(direction)]
    groups = list(dict.fromkeys(subset['group']))
    for index, group_name in enumerate(groups):
        values = subset.loc[subset['group'].eq(group_name), alpha_column]
        ax.scatter(np.full(len(values), index), values, color='#0072B2' if group_name.startswith('brains') else '#D55E00')
        ax.plot([index - 0.2, index + 0.2], [values.mean(), values.mean()], color='black')
    group_labels = [group.replace(' | ', '\n', 1) for group in groups]
    ax.set_xticks(range(len(groups)), group_labels, rotation=90, ha='right', va='center',
                  rotation_mode='anchor')
    ax.tick_params(axis='x', labelsize=8.5, pad=4)
    ax.set_title(direction)
    ax.set_ylabel(ylabel)
fig.tight_layout(pad=1.5, h_pad=3.0, w_pad=3.0)
if SAVE_OUTPUTS:
    save_figure_formats(fig, CROSS_OUTPUT_DIR / '13_brain_vs_selected_phantom_alpha.png', dpi=300)
plt.show()


In [ ]:
if SAVE_OUTPUTS:
    CROSS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    phantom_shape.to_csv(CROSS_OUTPUT_DIR / 'phantom_shape_metrics.csv', index=False)
    phantom_alpha.to_csv(CROSS_OUTPUT_DIR / 'phantom_alpha_harmonized.csv', index=False)
    phantom_alpha_internal.to_csv(CROSS_OUTPUT_DIR / 'phantom_alpha_internal_water.csv', index=False)
    comparison_shape.to_csv(CROSS_OUTPUT_DIR / 'brain_vs_phantom_shape_metrics.csv', index=False)
    comparison_alpha.to_csv(CROSS_OUTPUT_DIR / 'brain_vs_phantom_alpha_harmonized.csv', index=False)
    comparison_alpha_internal.to_csv(CROSS_OUTPUT_DIR / 'brain_vs_phantom_alpha_internal_reference.csv', index=False)
    model_audit.to_csv(CROSS_OUTPUT_DIR / 'shared_vs_separate_tc_model_audit.csv', index=False)
    model_audit_summary.to_csv(CROSS_OUTPUT_DIR / 'shared_vs_separate_tc_model_summary.csv', index=False)
    protocol_audit.to_csv(CROSS_OUTPUT_DIR / 'protocol_compatibility_audit.csv', index=False)
    print(f'Cross-dataset tables and figures saved under: {CROSS_OUTPUT_DIR}')
